# ENG-QUANT quick start

Choose one of the two workflows below:

- **toy**: build a small quadratic toy problem, create a linear approximation, and solve it.
- **real**: linearize an existing JSON problem, remap its nonzero lower bounds to zero, and solve it.

This notebook is intended to be run from the `ENG-QUANT-NEW/experiments` folder.

In [55]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
from eng_quant.generators import toy_problem_to_json_reduced
from eng_quant.relaxers import (
    quadratic_lin_approx_no_surrogate,
    remap_lower_bounds
)
from eng_quant.ipm import (
    ipm_solve,
    ipm_solve_fo,
    ipm_solve_so,
    ipm_solve_no_split,
    ipm_solve_no_split_fo,
    ipm_solve_no_split_so
)



## Input options

Set `PROBLEM_KIND` to `"toy"` or `"real"`. For a real problem, set `REAL_PROBLEM_FILE` to a JSON file in `ENG-QUANT-NEW/experiments`.

In [57]:
# Workflow: "toy" or "real"
PROBLEM_KIND = "toy"

# Toy-problem inputs
N = 2
F_UPPER_BOUNDS = True
DEMAND_INEQUALITY = False
GAMMA = 1.0
LAMBD = 1.0
EPS = 0.1
DELTA = 0.1
LP = LF = 0.0
UP = 0.5
CONVEX_RELAX = True
TOY_OUTPUT = "toy_quickstart.json"

# Real-problem inputs
REAL_PROBLEM_FILE = "model11_quad_reform.json"

# Linear approximation inputs
LIN_CONSTRAINTS = 10
OUTER_APPROXIMATION = True
ENDPOINTS = True
REMOVE_DIVISION = False

if PROBLEM_KIND not in {"toy", "real"}:
    raise ValueError("PROBLEM_KIND must be 'toy' or 'real'.")

In [58]:
# Build a toy problem or select a real problem.
if PROBLEM_KIND == "toy":
    toy_problem_to_json_reduced(
        N,
        F_UPPER_BOUNDS,
        DEMAND_INEQUALITY,
        GAMMA,
        LAMBD,
        EPS,
        DELTA,
        LP,
        LF,
        UP,
        convex_relax=CONVEX_RELAX,
        f_out=TOY_OUTPUT,
        verbose=True,
    )
    quadratic_input = TOY_OUTPUT
else:
    quadratic_input = REAL_PROBLEM_FILE

# Convert quadratic constraints into a linear-approximation JSON file.
quadratic_lin_approx_no_surrogate(
    LIN_CONSTRAINTS,
    OUTER_APPROXIMATION,
    remove_division=REMOVE_DIVISION,
    endpoints=ENDPOINTS,
    f_name=quadratic_input,
    verbose=True,
)

# Real models commonly contain nonzero lower bounds. The IPM requires 0.
if PROBLEM_KIND == "real":
    remap_lower_bounds(
        f_name="linear_approx.json",
        verbose=True,
        quadratic_check=True,
        lb=0,
    )

Variables: 11
Non-bound constraints: 11
Approximation points for quadratic variable f[production1_transient1]:
[0.         0.11111111 0.22222222 0.33333333 0.44444444 0.55555556 0.66666667 0.77777778 0.88888889 1.        ]
Approximation points for quadratic variable f[production2_transient1]:
[0.         0.07856742 0.15713484 0.23570226 0.31426968 0.3928371  0.47140452 0.54997194 0.62853936 0.70710678]
Approximation points for quadratic variable f[transient1_customer1]:
[0.         0.07856742 0.15713484 0.23570226 0.31426968 0.3928371  0.47140452 0.54997194 0.62853936 0.70710678]
Approximation points for quadratic variable f[production4_transient2]:
[0.         0.11111111 0.22222222 0.33333333 0.44444444 0.55555556 0.66666667 0.77777778 0.88888889 1.        ]
Approximation points for quadratic variable f[production5_transient2]:
[0.         0.07856742 0.15713484 0.23570226 0.31426968 0.3928371  0.47140452 0.54997194 0.62853936 0.70710678]
Approximation points for quadratic variable f[t

## IPM options

Set `SOLVER` to `"split"`, `"split_fo"`, `"split_so"`, `"no_split"`, `"no_split_fo"`, or `"no_split_so"`.

Start with `qsim=False` for the classical path where supported.

In [59]:
# Solver: "split", "split_fo", "split_so", "no_split", "no_split_fo", or "no_split_so"
SOLVER = "no_split_fo"

SOLVERS = {"split", "split_fo", "split_so", "no_split", "no_split_fo", "no_split_so"}
if SOLVER not in SOLVERS:
    raise ValueError(f"SOLVER must be one of {sorted(SOLVERS)}.")

IPM_INPUT = "linear_approx.json"
IPM_BETA = 0.1
IPM_XI = 1 - 5e-8
IPM_OMEGA = 1e4
IPM_EPSILON = 1e-8
IPM_INFEAS_FACTOR = 1000.0
SYMMETRIC_NEIGHBORHOOD = True
ZETA_N = 1e-2
ZETA_R = 0.3
ALPHA_SCHED = [0.9, 0.7, 0.5, 0.3, 0.1, 0.01]
ALPHA_MIN = 1e-3
SAUND_TOM = False
SAUND_TOM_FACTOR = 1e-16
SYMMETRIC_SCALING = True
OMEGA_BOUND = True
SPARSE = False
CLASSIC_ILU = False
QSIM = True
REFINEMENT_ITERS = 10
ILU_DROP_TOL = 1e-4
ILU_RTOL = 1e-6
HERMITIAN_EMBEDDING = False
BACKEND_NAME = "emulated"
DALZELL_INVERSION = False
HADAMARD_READOUT = False
IPM_VERBOSE = True
PRINT_QLSA_STATS = False
DRAW_QLSA = False
PHYS_EST_QLSA = False

In [60]:
base_ipm_options = {
    "f_name": IPM_INPUT,
    "xi": IPM_XI,
    "omega": IPM_OMEGA,
    "epsilon": IPM_EPSILON,
    "infeas_factor": IPM_INFEAS_FACTOR,
    "sym_neighborhood": SYMMETRIC_NEIGHBORHOOD,
    "zeta_n": ZETA_N,
    "zeta_r": ZETA_R,
    "alpha_sched": ALPHA_SCHED,
    "alpha_min": ALPHA_MIN,
    "saund_tom": SAUND_TOM,
    "saund_tom_factor": SAUND_TOM_FACTOR,
    "omega_bound": OMEGA_BOUND,
    "sparse": SPARSE,
    "classic_ilu": CLASSIC_ILU,
    "refinement_iters": REFINEMENT_ITERS,
    "ilu_drop_tol": ILU_DROP_TOL,
    "dalzell_inversion": DALZELL_INVERSION,
    "hadamard_readout": HADAMARD_READOUT,
    "verbose": IPM_VERBOSE,
    "print_qlsa_stats": PRINT_QLSA_STATS,
    "draw_qlsa": DRAW_QLSA,
    "phys_est_qlsa": PHYS_EST_QLSA,
}

solver_functions = {
    "split": ipm_solve,
    "split_fo": ipm_solve_fo,
    "split_so": ipm_solve_so,
    "no_split": ipm_solve_no_split,
    "no_split_fo": ipm_solve_no_split_fo,
    "no_split_so": ipm_solve_no_split_so,
}

solver_options = {
    "split": {"beta": IPM_BETA, "qsim": QSIM, "backend_name": BACKEND_NAME},
    "split_fo": {"qsim": QSIM, "backend_name": BACKEND_NAME},
    "split_so": {},
    "no_split": {
        "beta": IPM_BETA,
        "symmetric_scaling": SYMMETRIC_SCALING,
        "ilu_rtol": ILU_RTOL,
        "hermitian_embedding": HERMITIAN_EMBEDDING,
        "qsim": QSIM,
        "backend_name": BACKEND_NAME,
    },
    "no_split_fo": {
        "symmetric_scaling": SYMMETRIC_SCALING,
        "ilu_rtol": ILU_RTOL,
        "hermitian_embedding": HERMITIAN_EMBEDDING,
        "qsim": QSIM,
        "backend_name": BACKEND_NAME,
    },
    "no_split_so": {"symmetric_scaling": SYMMETRIC_SCALING},
}

solver_functions[SOLVER](**base_ipm_options, **solver_options[SOLVER])

Time to load matrices: 0.00 seconds.
8.091168824543143
omega was limited by the max cost
Cond no before precond: 7.906631839454957
Cond no before embedding: 1.0000000000000009
Encoded problem
Residual norm: 2.5688294621445143e-14

alpha_star_y_aff = 0.8539790265241018
alpha_star_q_aff = 0.9823193274623198
alpha_star_lam_aff = 0.9375962361893182
alpha_star_z1_aff = 0.703464588142532
alpha_star_z2_aff = 0.5052636426522323
alpha_star_z3_aff = 0.5179719455344259


alpha_star_y = 0.8409090295003497
alpha_star_q = 0.9792760320704004
alpha_star_lam = 0.9348233521873219
alpha_star_z1 = 0.6754764074650277
alpha_star_z2 = 0.474778671866539
alpha_star_z3 = 0.4859827287906917

The step size was 0.4273008046798851
Iteration 1:
Primal objective:        2.44424676e+02 
Dual objective:          -1.93864838e+00

Norm of Primal residual: 1.36e+02
Norm of Dual residual:   4.19e+01
Abs. Compl. Gap:         3.65e+03
Rel. Compl. Gap:         1.48e+01

Cond no before precond: 19.377852060092785
Cond no befor

(np.float64(2.2191718112255967),
 1.2826788425445557,
 array([0.25308642, 0.25308641, 0.5       , 0.00635544, 0.59585905, 0.50414095, 0.50000001, 0.00192644, 0.59585906, 0.50414094, 0.00221449]),
 np.float64(8.410051474056637e-09))